In [1]:
from pathlib import Path

MODEL_DIR  = Path("model/BD_llama_6heads_1epoch_4layers")
DATA_DIR   = Path("data/BD_llama_inital")
REMAP_PATH = DATA_DIR / "old_to_new.json"
TOKENS_PATH = DATA_DIR / "bios_postreduce.bin"

In [2]:
from condensed_tokenizer import CondensedTokenizer
from bio_sampler import BioSampler

tokenizer = CondensedTokenizer.from_remap_path(REMAP_PATH)
sampler   = BioSampler(DATA_DIR / "people.json", fields=("birthday",), seed=0)

print(f"vocab_size = {tokenizer.vocab_size}, eos_token_id = {tokenizer.eos_token_id}")
print(f"{len(sampler.people):,} people, {sampler.n_templates} templates/person\n")

# Specific person + specific template
text = sampler.render(sampler.people[0], exposure_idx=0)
print("render(people[0], 0) →", repr(text))
print("encode →", tokenizer.encode(text)[:15], "...\n")

# Random bio
draw = sampler.sample()
print(f"sample() → person id={draw['person']['id']}, template={draw['exposure_idx']}")
print("text →", repr(draw["text"]))

/Users/efmac/Code/Project Code/CRL-Interp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


vocab_size = 1836, eos_token_id = 1835
50,000 people, 46 templates/person

render(people[0], 0) → ' Gabriella Ella Rigby was born on February 18, 1816.'
encode → [870, 83, 882, 663, 5, 1273, 267, 80, 536, 52, 487, 237, 1, 237, 256] ...

sample() → person id=50494, template=26
text → ' Marco Jackson Rowland arrived in this world on December 24, 1717, a day to be remembered.'


In [3]:
import torch
from transformers import LlamaForCausalLM
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.loading_from_pretrained import convert_llama_weights


def pick_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = pick_device()
dtype = torch.float32

hf_model = LlamaForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=dtype)
hf_model.eval()
assert hf_model.config.vocab_size == tokenizer.vocab_size, (
    f"checkpoint vocab {hf_model.config.vocab_size} != remap vocab "
    f"{tokenizer.vocab_size} — wrong old_to_new.json for this model."
)

# Build the TL config from the HF config so dims match our custom
# 4-layer / hidden=384 / vocab=1836 model (from_pretrained would have used
# the Llama-2-7b template config and tried to read layer 4 of a 4-layer model).
hf_cfg = hf_model.config
tl_cfg = HookedTransformerConfig(
    n_layers = hf_cfg.num_hidden_layers,
    d_model = hf_cfg.hidden_size,
    d_head = hf_cfg.hidden_size // hf_cfg.num_attention_heads,
    n_heads = hf_cfg.num_attention_heads,
    d_mlp= hf_cfg.intermediate_size,
    d_vocab=hf_cfg.vocab_size,
    n_ctx=hf_cfg.max_position_embeddings,
    act_fn="silu",
    normalization_type="RMS",
    gated_mlp=True,
    positional_embedding_type="rotary",
    rotary_base=int(getattr(hf_cfg, "rope_theta", 10000.0)),
    rotary_dim=hf_cfg.hidden_size // hf_cfg.num_attention_heads,
    final_rms=True,
    tie_word_embeddings=hf_cfg.tie_word_embeddings,
    initializer_range=hf_cfg.initializer_range,
    n_key_value_heads=hf_cfg.num_key_value_heads,
    device=device,
)

# Pre-tokenize everything you feed the model; don't attach the tokenizer.
# TL only calls into the tokenizer for `model(str)` / `to_tokens` / `to_string`,
# none of which we use — we always pass token ids directly.
state_dict = convert_llama_weights(hf_model, tl_cfg)
model = HookedTransformer(tl_cfg)
model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()
print(f"Loaded on {device}: n_layers={model.cfg.n_layers}, d_model={model.cfg.d_model}, "
      f"n_heads={model.cfg.n_heads}, d_vocab={model.cfg.d_vocab}")

Loading weights: 100%|██████████| 38/38 [00:00<00:00, 34239.22it/s]

Moving model to device:  mps
Loaded on mps: n_layers=4, d_model=384, n_heads=6, d_vocab=1836



/Users/efmac/Code/Project Code/CRL-Interp/.venv/lib/python3.13/site-packages/transformer_lens/config/HookedTransformerConfig.py:354: UserWarning: MPS backend may produce silently incorrect results (PyTorch 2.12.0). Set TRANSFORMERLENS_ALLOW_MPS=1 to suppress this warning. See: https://github.com/TransformerLensOrg/TransformerLens/issues/1178
  warn_if_mps(self.device)


In [4]:
from data_loader import make_dataloader

loader = make_dataloader(TOKENS_PATH, seq_len=512, batch_size=4, shuffle=False)
print(f"{len(loader.dataset):,} sequences, {len(loader):,} batches")

batch = next(iter(loader))
print({k: tuple(v.shape) for k, v in batch.items()})
print("first sequence decoded:")
print(tokenizer.decode(batch["input_ids"][0][:80]))

def show_tokens(ids, tokenizer, addOne=False):
    """Print each token with its position and decoded text (with repr so
    leading spaces / newlines stay visible).

    addOne=True shifts the displayed index by 1, so positions match what
    the model sees after a BOS/EOS is prepended to `ids` downstream.
    """
    if hasattr(ids, "tolist"):
        ids = ids.tolist()
    if ids and isinstance(ids[0], list):
        ids = ids[0]   # unwrap [1, N] batch

    offset = 1 if addOne else 0
    id_w = max(len(str(t)) for t in ids)
    idx_w = len(str(len(ids) - 1 + offset))
    print(f"{'idx':>{idx_w}} | {'id':>{id_w}} | text")
    print("-" * (idx_w + id_w + 12))
    for i, t in enumerate(ids):
        print(f"{i + offset:>{idx_w}} | {t:>{id_w}} | {tokenizer.decode([t])!r}")


175,783 sequences, 43,946 batches
{'input_ids': (4, 512), 'labels': (4, 512)}
first sequence decoded:
 Grace Caroline Hancock was brought into existence on April 11, 1711. Veronica Daisy Long celebrates their special day each year on March 10, 1872. Ana Eden Spence rejoices on March 15, 1850, the day they were born. Jennifer Kaitlyn Patrick's birth date is October 13, 1718. Leonardo Xavier Pritchard acknowledges June 23, 1889 as the


### SAE

In [5]:
from sae_lens import SAE, HookedSAETransformer, LoggingConfig
from sae_lens import LanguageModelSAERunnerConfig, LanguageModelSAETrainingRunner, JumpReLUTrainingSAEConfig

In [10]:
dataset

NameError: name 'dataset' is not defined

In [12]:
import numpy as np
from datasets import Dataset

# bios_postreduce.bin is a flat uint16 stream of reduced-vocab token ids —
# already tokenized and vocab-remapped (see data_loader.PackedTokenDataset).
# SAE Lens's ActivationsStore wants a HuggingFace Dataset with an `input_ids`
# column; reshape the flat stream into fixed-length rows and wrap it. Passing
# this object to the runner as `override_dataset=` skips load_dataset and any
# disk round-trip — the store inspects one row, sees `input_ids`, and treats
# the dataset as pre-tokenized.
#
# context_size must be <= the model's n_ctx (512) — it can't attend past 512
# positions — so we pack the SAE's sequences at 512, matching the .bin.
context_size  = 512

_flat   = np.memmap(TOKENS_PATH, dtype=np.uint16, mode="r")
_n_seq  = len(_flat) // context_size
_tokens = np.asarray(_flat[: _n_seq * context_size]).reshape(_n_seq, context_size)

# int32: token ids (0..1835) fit easily; avoids uint16 dtype surprises in Arrow/torch.
sae_dataset = Dataset.from_dict({"input_ids": _tokens.astype(np.int32)})

_row = next(iter(sae_dataset))
print(f"sae_dataset: {len(sae_dataset):,} sequences x {context_size} tokens "
      f"= {len(sae_dataset) * context_size:,} tokens")
print(f"columns={sae_dataset.column_names}  row_len={len(_row['input_ids'])}  "
      f"id_range=[{_tokens.min()}, {_tokens.max()}]  (model d_vocab={model.cfg.d_vocab})")


sae_dataset: 175,783 sequences x 512 tokens = 90,000,896 tokens
columns=['input_ids']  row_len=512  id_range=[0, 1835]  (model d_vocab=1836)


In [26]:
#trainingParms
batch_size = 4096

total_training_tokens = 90_000_896
total_training_steps = (total_training_tokens // batch_size)

#### Setup Dataset

##### SAE CONFIG

In [ ]:
cfg = LanguageModelSAERunnerConfig(
    sae=JumpReLUTrainingSAEConfig(
        l0_coefficient=5.0, # Sparsity penalty coefficient
        jumprelu_sparsity_loss_mode="tanh",
        jumprelu_tanh_scale=4.0, # default value
        jumprelu_bandwidth=2.0,
        jumprelu_init_threshold=0.1,
        pre_act_loss_coefficient=3e-6,
        # Anthropic's settings assume normalized activations
        normalize_activations="expected_average_only_in",
        l0_warm_up_steps= (total_training_steps//50),
        d_in=model.cfg.d_model, # must match your hook point
        d_sae=model.cfg.d_model * 16,
    ),
    # Data generation (Model + training distribiton)
    model_name="", 
    hook_name="blocks.1.hook_mlp_out",
    dataset_path="",  # tokenized language dataset.
    is_dataset_tokenized=True,
    prepend_bos=True,  # you should use whatever the base model was trained with
    streaming=True,  # we could pre-download the token dataset if it was small.
    train_batch_size_tokens=batch_size,
    context_size=context_size,
    #
    # Activations store
    n_batches_in_buffer=64,
    training_tokens=total_training_tokens,
    store_batch_size_prompts=16,
    #
    # Training hyperparameters (standard)
    lr=5e-5,
    adam_beta1=0.9,
    adam_beta2=0.999,
    lr_scheduler_name="constant",  # controls how the LR warmup / decay works
    lr_warm_up_steps= (total_training_steps//50),  # avoids large number of initial dead features
    lr_decay_steps=(total_training_steps//4),  # helps avoid overfitting
    # Training hyperparameters (resampling)
    feature_sampling_window=2000,  # how often we resample dead features
    dead_feature_window=1000,  # size of window to assess whether a feature is dead
    dead_feature_threshold=1e-4,  # threshold for classifying feature as dead, over window
    # Logging / evals
    logger=LoggingConfig(
        log_to_wandb=True, 
        wandb_project="interpLM4",
        wandb_log_frequency=30,
        eval_every_n_wandb_logs=20,
    ),
    # Misc.
    device=str(device),
    seed=42,
    n_checkpoints=5,
    checkpoint_path="sae_runs/checkpoints",
    dtype="float32",
)


In [28]:
runner = LanguageModelSAETrainingRunner(
    cfg,
    override_dataset=sae_dataset,   # <- the wrapped .bin
    override_model=model,           # <- your already-loaded HookedTransformer
)

You just passed in a dataset which will override the one specified in your configuration: . As a consequence this run will not be reproducible via configuration alone.
You just passed in a model which will override the one specified in your configuration: . As a consequence this run will not be reproducible via configuration alone.
/Users/efmac/Code/Project Code/CRL-Interp/.venv/lib/python3.13/site-packages/sae_lens/training/activations_store.py:358: UserWarning: The training dataset contains fewer samples (175783) than the number of samples required by your training configuration (90000896). This will result in multiple training epochs and some samples being used more than once.
  warnings.warn(


### Metrics

In [ ]:
#Reconsturction Loss: MSE Loss, CE Los recovered, Explained varinavce
# Sparisty: L0, L1 Staitics and hustogram


### Analysis of SAE

In [ ]:
dataset = load_dataset(cfg.dataset_path, streaming=True)
batch_size = 1024
tokens = t.tensor(
    [x["input_ids"] for i, x in zip(range(batch_size), dataset["train"])],
    device=str(device),
)
print(tokens.shape)
sae_vis_data = SaeVisData.create(
    sae=model,
    model=model,
    tokens=tokens,
    cfg=SaeVisConfig(features=range(16)),
    verbose=True,
)
sae_vis_data.save_feature_centric_vis(
    filename=str(section_dir / "feature_vis.html"),
    verbose=True,
)



